# Gun 3 - Dogruluk Testleri

Belge -> JSON cikarim hattinin dogrulugunu test belgeleri uzerinde olcer.

In [ ]:
import os
import json
import base64
import glob
import re
from dotenv import load_dotenv
import anthropic
import pandas as pd

load_dotenv(dotenv_path='../.env')
api_key = os.getenv("ANTHROPIC_API_KEY")
if not api_key:
    raise RuntimeError("API anahtari bulunamadi. .env dosyani kontrol et.")

client = anthropic.Anthropic(api_key=api_key)
print("API anahtari yuklendi, client hazir.")

## 1. Extraction fonksiyonu

In [ ]:
SYSTEM_PROMPT = """Sen bir dokuman analiz asistanisin. Sana resmi bir talep formunun gorseli verilecek.

Gorevin, belgedeki bilgileri SADECE asagidaki JSON semasina uygun sekilde cikarmaktir:

{
  "talep_eden": string,
  "tarih": string,        // YYYY-MM-DD formatinda normalize edilmis tarih
  "departman": string,
  "konu": string,
  "aciklama": string
}

KURALLAR:
- Yanitin SADECE gecerli bir JSON nesnesi olmali.
- Markdown kod blogu (uc backtick), aciklama cumlesi veya baska hicbir metin EKLEME. Yanitin '{' ile baslayip '}' ile bitmeli.
- Semadaki tum alanlari doldur. Bir bilgi belgede yoksa degerini null yap.
- Belgede olmayan bilgi UYDURMA.
"""

USER_INSTRUCTION = "Bu belgeyi yukaridaki semaya gore analiz et ve JSON olarak dondur."


def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')


def extract_json(text: str) -> dict:
    cleaned = text.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.strip("`")
        if cleaned.startswith("json"):
            cleaned = cleaned[4:]
        cleaned = cleaned.strip()
    return json.loads(cleaned)


def extract_document(image_path: str) -> dict:
    """Bir belge gorselini Claude'a gonderir ve semaya uygun JSON dondurur.

    Basarisiz JSON parse durumunda {'_error': ..., '_raw': ...} dondurur
    (calisan dongu tek belgede takilip kalmasin diye exception firlatmaz).
    """
    base64_image = encode_image(image_path)
    response = client.messages.create(
        model="claude-sonnet-5",
        max_tokens=1024,
        system=SYSTEM_PROMPT,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "source": {
                            "type": "base64",
                            "media_type": "image/png",
                            "data": base64_image,
                        },
                    },
                    {"type": "text", "text": USER_INSTRUCTION},
                ],
            }
        ],
    )
    raw_text = "".join(block.text for block in response.content if block.type == "text")
    try:
        return extract_json(raw_text)
    except json.JSONDecodeError as e:
        return {"_error": str(e), "_raw": raw_text}


print("extract_document() hazir.")

## 2. Karsilastirma / skor fonksiyonu

In [ ]:
FIELDS = ["talep_eden", "tarih", "departman", "konu", "aciklama"]


def normalize(value):
    if value is None:
        return None
    return re.sub(r"\s+", " ", str(value).strip())


def score_document(predicted: dict, expected: dict) -> dict:
    """Tek bir belge icin alan bazinda dogru/yanlis sonucunu dondurur."""
    result = {}
    for field in FIELDS:
        pred_val = normalize(predicted.get(field))
        exp_val = normalize(expected.get(field))
        result[field] = (pred_val == exp_val)
    result["exact_match"] = all(result[f] for f in FIELDS)
    result["field_accuracy"] = sum(result[f] for f in FIELDS) / len(FIELDS)
    return result


print("score_document() hazir.")

## 3. Test setini calistir

In [ ]:
GT_PATH = "../data/processed/ground_truth.json"
with open(GT_PATH, encoding="utf-8") as f:
    ground_truth = json.load(f)

rows = []
predictions = {}

for filename, expected in sorted(ground_truth.items()):
    image_path = os.path.join("..", "data", "raw_docs", filename)
    print(f"Isleniyor: {filename} ...")
    predicted = extract_document(image_path)
    predictions[filename] = predicted

    if "_error" in predicted:
        row = {"belge": filename, "exact_match": False, "field_accuracy": 0.0, "hata": predicted["_error"]}
        for field in FIELDS:
            row[field] = False
    else:
        scores = score_document(predicted, expected)
        row = {"belge": filename, **scores, "hata": None}
    rows.append(row)

print("\nTum belgeler islendi.")

## 4. Sonuclari ozetle

In [ ]:
results_df = pd.DataFrame(rows)
cols = ["belge"] + FIELDS + ["exact_match", "field_accuracy", "hata"]
results_df = results_df[cols]
results_df

In [ ]:
overall_field_accuracy = results_df["field_accuracy"].mean()
exact_match_rate = results_df["exact_match"].mean()

print(f"Genel alan-bazli dogruluk: {overall_field_accuracy:.1%}")
print(f"Tam eslesme (exact match) orani: {exact_match_rate:.1%}")
print()
print("Alan bazinda dogruluk:")
for field in FIELDS:
    acc = results_df[field].mean()
    print(f"  {field}: {acc:.1%}")

## 5. Hatali alanlarin detayini incele

In [ ]:
for filename, expected in sorted(ground_truth.items()):
    predicted = predictions[filename]
    if "_error" in predicted:
        continue
    for field in FIELDS:
        pred_val = normalize(predicted.get(field))
        exp_val = normalize(expected.get(field))
        if pred_val != exp_val:
            print(f"[{filename}] {field}")
            print(f"  Beklenen : {exp_val!r}")
            print(f"  Alinan   : {pred_val!r}")
            print()